In [25]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots


- Volatility, is deviation from an expectation/benchmark and so far in the notebooks treated as
    - return = percentage change of price
    - standart deviation of return = volatility

- With this (coming from the definition of standart dev),  have taken average return as our benchmark.


### Forward, Spot and Implied Volatility

* Spot (historical) volatility
    * The volatility estimate prevailing right now, typically computed from realized (historical) returns.
    * Sensitive to the **lookback window (10 day, 30 day, 252 day) that has been chosen**

* Forward volatility
    * the expected volatility over a specific future period
    * For example, "what will vol look like starting 3 months from now, over the following 3 months?" It's a forward-looking expectation **priced into today's market**, not directly observable.

* Implied Volatility
    * volatility **back-solved** from option prices. If the market prices an option at some level, you invert Black-Scholes to ask "what vol assumption is consistent with this price?"
    * IV therefore reflects the **market's consensus expectation about future vol**,

### Volatility Conversion (daily to monthly, daily to annually)

Volatility scales with the square root of time, not linearly.

$$\sigma_{\text{monthly}} = \sqrt{21} \cdot \sigma_d \qquad \sigma_{\text{annual}} = \sqrt{252} \cdot \sigma_d$$

where 21 and 252 are the assumed number of trading days per month and year respectively. Note these are *trading day* conventions -- calendar days are not used because markets are closed on weekends and holidays.

### Unobservability of Forward Looking Variance and Expected Returns

* If we take expected returns as our benchmark, and define the variance from that as our volatility:

Expected returns (forward looking):

  $$\mathbb{E}[R_t] = \mu_t$$

Variance of returns (forward looking):

  $$\text{Var}(R_t) = \mathbb{E}[(R_t - \mu_t)^2] = \sigma_t^2$$

- Mut is not observeable (its an expecetation)
- Mut might change over time
- **An alternative -what we have done so far in previous notebooks- : **USING SPOT/HISTORICAL RETURNS AS A PROXY**
    * mean returns were $\mu_t$ for us.
- We will discuss different approaches in this notebook

### Volatility Clustering and Rolling Volatility (VIX, Variance Risk Premium)

* Rolling Volatility
    * The idea is straightforward: instead of computing a single volatility estimate over the entire sample, you fix a lookback window of length and slide it forward one observation at a time.
    * Short window (e.g. 10-day): reacts quickly to new information, but very noisy. A single large return move dominates the estimate.
    * Long window (e.g. 252-day): smooth and stable, but slow to react. By the time a volatility regime change shows up in the estimate, it may already be over.


* Why it matters / Volatility Clustering
    * Volatility is not constant, it clusters. Calm periods are followed by calm periods, turbulent periods are followed by turbulent periods. Or *high changes will be followed by high changes, low changes will be followed by low changes*
        * Rolling volatility makes this visible. On a 30-day window, crises/spikes after calm regimes can be seen: 2008, the 2020 COVID crash, the 2022 rate shock.

#### VIX and Variance Risk Premium

* VIX Benchmark
    * VIX is the market's 30-day implied volatility on the S&P 500, derived from option prices across a range of strikes. It is often called the "fear index" because it spikes sharply when markets sell off -- options become expensive as demand for downside protection surges.

* Relationship between VIC and **ROLLING REALIZED VOL**
    * **VIX is almost always above realized vol**
        *  This spread is called the variance risk premium. Investors pay a premium for protection, so implied vol consistently overshoots realized vol on average.
    * During crises, **VIX spikes before realized vol catches up**, because options reprice instantly while rolling realized vol is anchored to the past window.
    * After a shock, **VIX mean-reverts faster than rolling realized vol**, because the window keeps the crisis observations in the estimate until they roll off.


### Stylized Facts of Volatility


#### **1. Volatility Clustering**

* Large return shocks tend to be followed by large shocks, and small shocks by small shocks, regardless of sign.
    * Volatility arrives in regimes rather than being randomly distributed through time.
    * Formally, squared and absolute returns exhibit significant positive ""autocorrelation"":

$$\text{Corr}(|r_t|, |r_{t-k}|) > 0 \quad \text{for large } k$$

This is the most fundamental stylized fact in empirical finance. **It directly invalidates the constant volatility assumption underlying Black-Scholes and standard OLS, and serves as the primary motivation for the GARCH family of models, which explicitly parameterize this persistence through the conditional variance equation.**



####  **2. Mean Reversion**

Although volatility exhibits persistence, **it does not diverge indefinitely**. It gravitates toward a long-run equilibrium level over time -- extreme regimes, both high and low, are transient.

In GARCH(1,1), this behavior is captured through the relationship between conditional variance and unconditional (long-run) variance. The model's conditional variance equation is:

\begin{equation}
    \sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2
\end{equation}

The unconditional (long-run) variance implied by this is:

\begin{equation}
    \bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}
\end{equation}

Whenever today's conditional variance $\sigma_t^2$ is above or below this long-run level $\bar{\sigma}^2$, the model structure pulls it back toward $\bar{\sigma}^2$ over subsequent periods. How fast this pull happens depends on $\alpha + \beta$ — the closer this sum is to 1, the slower the reversion (more persistence); the further from 1, the faster volatility snaps back to its long-run average.


####  * **The Practical Consequence: VIX Term Structure**
    * In **calm** markets, the VIX curve plotted across maturities is usually upward-sloping
        * **longer-dated contracts price in higher volatility than shorter-dated ones.**
                * the market is calm right now (low volatility), but it knows volatility will likely revert upward toward its long-run level over time, so longer-horizon pricing reflects that expected reversion.


#### **3. Leverage Effect**

* **Negative return shocks generate a disproportionately larger increase in subsequent volatility than positive shocks of equivalent magnitude**.

* The relationship between returns and volatility is asymmetric.

* Two mechanisms are commonly proposed.
    * Under the financial leverage channel, a **decline in equity value raises** the firm's debt-to-equity ratio, increasing the **riskiness of equity and thus its return volatilit**y.
    * Under the risk premium channel, falling prices elevate investor risk aversion, **raising required returns and further depressing prices**, creating a feedback loop between prices and volatility.

* Empirically, the leverage effect manifests in the implied volatility surface as negative skew:
    * out-of-the-money  put options(STRIKE PRICE < CURRENT VALUE OF THE STOCK)  command a higher implied volatility than out-of-the-money calls at the same distance from the forward.

Standard symmetric GARCH does not accommodate this feature; asymmetric extensions such as GJR-GARCH and EGARCH were developed specifically to address it.



####  **4. Heavy Tails**

**Empirical return distributions exhibit excess kurtosis relative to the Gaussian benchmark.**
Extreme realizations occur with a frequency that normal distribution assumptions **systematically understate.**

This has direct and material consequences for risk measurement. VaR and CVaR estimated under a normality assumption will underestimate tail risk, particularly at high confidence levels. Heavy-tailed alternatives -- Student-$t$, generalized hyperbolic, or stable distributions -- are required for adequate tail fit. This issue is directly relevant to the parametric vs. historical VaR discrepancy encountered in the QRM project: the normal distribution misallocated probability mass between the center and the tails relative to the empirical distribution, producing the analytically interesting crossing of the two estimates.



**5. Long Memory**

The autocorrelation function of squared or absolute returns decays hyperbolically rather than exponentially. Past volatility exerts statistically significant influence on future volatility across lags of weeks or months, well beyond what short-memory models such as GARCH can reproduce.

This property is distinct from clustering, which describes short-run persistence. Long memory refers to a slower, power-law decay in autocorrelation that implies a degree of fractional integration in the variance process. Modeling frameworks proposed to capture this include ARFIMA specifications applied to realized variance, and rough volatility models based on fractional Brownian motion with Hurst exponent $H < 0.5$. The empirical evidence, however, remains contested: a strand of the literature argues that apparent long memory in volatility may be a statistical artifact of structural breaks or regime changes rather than a genuine feature of the data-generating process.

# Historical Volatility and Clustering

In [14]:
tickers = ["AAPL", "JPM", "XOM", "KO"]

close_price = pd.read_csv("../data_general/4_ticker_data.csv")
close_price = close_price.set_index("Date")

percent_return = close_price.pct_change()
percent_return.head(2)



,AAPL,JPM,KO,XOM
Date,,,,
2020-01-02,NaN,NaN,NaN,NaN
2020-01-03,-0.009722,-0.013197,-0.005456,-0.008039


## Calculating Daily Volatility with Rolling Window

In [20]:
daily_volatility_10day = percent_return.rolling(window=10).std()

daily_volatility_30day = percent_return.rolling(window=30).std()
daily_volatility_252day = percent_return.rolling(window=30).std()



daily_volatility_10day.head(12)

,AAPL,JPM,KO,XOM
Date,,,,
2020-01-02,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN
2020-01-06,NaN,NaN,NaN,NaN
2020-01-07,NaN,NaN,NaN,NaN
2020-01-08,NaN,NaN,NaN,NaN
2020-01-09,NaN,NaN,NaN,NaN
2020-01-10,NaN,NaN,NaN,NaN
2020-01-13,NaN,NaN,NaN,NaN
2020-01-14,NaN,NaN,NaN,NaN


## Annualizing the Daily Volatility

Window size and scale are different things.

Window size determines how many days we consider to calculate the volatility at time x,
annualizing by multipliying sqr(252) enables us to talk about "yearly volatility of X percent

"If AAPL's 30-day annualized volatility comes out to 25%, it means: 'based on the return fluctuations observed over the last 30 days, if this stock kept moving at this same pace throughout the year, we'd expect an annualized dispersion of 25%'"



In [23]:
annualized_vol_10day = daily_volatility_10day * np.sqrt(252)
annualized_vol_30day = daily_volatility_30day * np.sqrt(252)
annualized_vol_252day = daily_volatility_252day * np.sqrt(252)

In [30]:
fig = go.Figure()

dataframes_to_plot = {

    "annualized_vol_10day" : annualized_vol_10day,
    "annualized_vol_30day" : annualized_vol_30day,
    "annualized_vol_252day" : annualized_vol_252day,

}

colors = ['#FF3131', '#00FFFF', '#39FF14', '#FF00FF', '#FFD700']


trace_tickers = []

for label, df in dataframes_to_plot.items():
    for ticker in tickers:
        fig.add_trace(go.Scatter(x=df.index, y=df[ticker], name=label + "_" + ticker))
        trace_tickers.append(ticker)

buttons = []

buttons.append(dict(
    label="All",
    method="update",
    args=[{"visible": [True] * len(trace_tickers)}]
))

for ticker in tickers:
    visibility = [tt == ticker for tt in trace_tickers]
    buttons.append(dict(
        label=ticker,
        method="update",
        args=[{"visible": visibility}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            buttons=buttons,
            x=0.05, y=1.15,
            showactive=True
        )
    ])


fig.show()

# Implied Volatility

The Black-Scholes model gives us a forward looking measure of volatility called implied volatility

Essentially, volatility is an input into an option's price and we can use market prices (generated by supply and demand) to produce this value

It is based on  gives us a sense of what level of volatility traders are pricing options expiring in the future at
